# Explore Chat Completions with Azure OpenAI

This notebook demonstrates how to call an Azure OpenAI chat completion model (`gpt-5.4-mini`) from Python.

It covers:

1. Installing the SDK
2. Setting up the Azure OpenAI client
3. Sending a single chat completion request
4. Maintaining chat history across multiple turns
5. Basic input validation and error handling

Requirements: Python 3.12.10 or later, VS Code with the Jupyter extension.

## Step 1: Install the OpenAI SDK

Run this once per environment. If you already have it installed, you can skip this cell.

> **Note:** The first installation may take approximately **1.5–2 minutes**.

In [ ]:
# Installs the OpenAI Python SDK (includes the Azure client)
%pip install --upgrade openai

## Step 2: Import dependencies

In [ ]:
# Azure-specific OpenAI client
from openai import AzureOpenAI

## Step 3: Configure your Azure OpenAI connection

Please paste your own key below.

> Note: For this local training exercise, you may temporarily use a key directly in the notebook. However, avoid hardcoding API keys in shared or production notebooks. Never commit a notebook containing a real API key to a public repository.

In [ ]:
azure_oai_endpoint = "https://hakunamatata1234.openai.azure.com/openai/v1"

# ToDo replace with your actual api key here
azure_oai_key = "<place_your_key_here>"

# Deployment name is fixed for this lab
azure_oai_deployment = "gpt-5.4-mini"

# Fails fast with a clear message instead of a confusing error later on
if not azure_oai_endpoint or "<" in azure_oai_endpoint or not azure_oai_endpoint.startswith("https://"):
    raise ValueError("Replace the placeholder endpoint above with your real Azure OpenAI endpoint before continuing.")

if not azure_oai_key or "<" in azure_oai_key or len(azure_oai_key) < 10:
    raise ValueError("Replace the placeholder key above with your real Azure OpenAI key before continuing.")

print("Configuration loaded.")

## Step 4: Initialize the Azure OpenAI client

In [ ]:
# Creates the client used for every request in this notebook
client = AzureOpenAI(
    azure_endpoint=azure_oai_endpoint,
    api_key=azure_oai_key,
    api_version="2024-12-01-preview",  # supports newer model deployments; adjust if your resource pins an older GA version
)

## Step 5: Send a single chat completion request

This is the core demo: a single call to the model with a short conversation, showing how system, user, and assistant roles work together.

In [ ]:
# Sends a one-off chat completion request with a short example conversation
response = client.chat.completions.create(
    model=azure_oai_deployment,
    temperature=0,
    max_completion_tokens=100,
    messages=[
        {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
        {"role": "user", "content": "Do penguins like pizzas?"},
        {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."},
        {"role": "user", "content": "Do polar bears support pizzas too?"},
    ],
)

In [ ]:
# Extracts the reply text and guards against an empty response
generated_text = response.choices[0].message.content if response.choices else None

if not generated_text:
    print("No response was returned by the model.")
else:
    print("Response: " + generated_text + "\n")

## Step 6: Maintain chat history

To hold a real conversation, keep a running list of messages and append every user and assistant turn to it before the next request.

In [ ]:
# Starts the conversation history with a system prompt and one prior exchange
messages_array = [
    {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
    {"role": "user", "content": "Do penguins like pizzas?"},
    {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."},
]

In [ ]:
# Small helper that sends the current history to the model and returns the reply
def ask_model(messages, temperature=0.7, max_completion_tokens=100):
    response = client.chat.completions.create(
        model=azure_oai_deployment,
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        messages=messages,
    )

    if not response.choices:
        return None

    return response.choices[0].message.content

## Step 7: Install Gradio

Install Gradio, a lightweight Python library for quickly creating web-based user interfaces.

> **Note:** The first installation may take approximately **1.5–3 minutes**.



In [ ]:
# Note: The first installation may take approximately **1.5–3 minutes**.

%pip install --upgrade gradio

## Step 8: Polished web UI with Gradio (with live history)

Gradio launches a local web chat UI that uses the same `ask_model()` function underneath. Previous chat history from the notebook is loaded into the UI, and new messages are added to the conversation as you chat. The updated history is also displayed live in the notebook output after each message.

In [ ]:
import gradio as gr


# Load the existing conversation history from previous notebook cells
initial_history = [
    {
        "role": message["role"],
        "content": message["content"]
    }
    for message in messages_array
    if message["role"] in ["user", "assistant"]
]


# Display previous notebook history
for message in initial_history:
    if message["role"] == "user":
        print("User: " + message["content"])
    elif message["role"] == "assistant":
        print("AI: " + message["content"])

print()


def gradio_chat(user_message, history):
    if not user_message or not user_message.strip():
        return "Please type a message before sending."

    if user_message.strip().lower() in ["quit", "exit"]:
        return (
            "Session ended. "
            "You may close this browser tab or refresh the page."
        )

    conversation = [
        {
            "role": "system",
            "content": "You are a helpful assistant with a lot of humor."
        }
    ]

    # Add previous chat history
    for message in history:
        content = message["content"]

        if isinstance(content, list):
            text_parts = []

            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    text_parts.append(block.get("text", ""))

            content = "".join(text_parts)

        if content:
            conversation.append({
                "role": message["role"],
                "content": content
            })

    # Add current user message
    conversation.append({
        "role": "user",
        "content": user_message
    })

    # Add user message to the existing notebook history
    messages_array.append({
        "role": "user",
        "content": user_message
    })

    # Send request to the Azure OpenAI model
    generated_text = ask_model(
        conversation,
        temperature=0.7,
        max_completion_tokens=100,
    )

    if not generated_text:
        return "No response was returned by the model."

    # Add generated response to the existing notebook history
    messages_array.append({
        "role": "assistant",
        "content": generated_text
    })

    # Update notebook output
    print("User: " + user_message)
    print("AI: " + generated_text)
    print()

    return generated_text


with gr.Blocks(
    title="Azure AI Chat"
) as demo:

    gr.Markdown(
        """
<div style="max-width: 900px; margin: 0 auto;">

# Azure AI Chat

Welcome to Azure AI Chat.

### Notes

- Previous chat history is loaded automatically.
- New messages are added to the existing chat history.
- Type **exit** or **quit** to end the session.
- Refresh the page to start a new Gradio conversation.

</div>
"""
    )

    chatbot = gr.Chatbot(
        value=initial_history
    )

    gr.ChatInterface(
        fn=gradio_chat,
        chatbot=chatbot,
        textbox=gr.Textbox(
            placeholder="Type a message... (type 'exit' to end the session)",
            container=True
        )
    )


print("\nOpen the chat UI using the local URL below")

demo.launch(
    inline=False,
    share=False,
    theme=gr.themes.Soft()
)

## Conclusion

Congratulations! You have successfully explored how to interact with an Azure OpenAI model using chat completions and how conversation history can be maintained across multiple interactions.

In this lab, you learned how to:

- Configure and authenticate an Azure OpenAI client.
- Send chat completion requests and process model responses.
- Maintain conversation history across multiple interactions.
- Run and interact with a Gradio-based chat application.

The concepts covered in this lab form the foundation of many real-world AI applications, including chatbots, copilots, and AI assistants that rely on conversation history to provide contextual responses.

As a best practice, avoid storing API keys directly in code. In production applications, store secrets securely using environment variables or a `.env` file and exclude them from source control.

> [!TIP]
> Finished early? Try the optional challenge activities below for additional hands-on practice.

### Optional Challenge Activities

1. Modify the application so that entering `quit` or `exit` closes the Gradio session.
2. Save and reload conversation history from a file.
3. Experiment with different system prompts to change the assistant's behaviour.
4. Add a feature to clear chat history and start a new conversation.

> [!NOTE]
> These activities are optional and intended for participants who complete the lab ahead of schedule.